In [1]:
%pip install numpy pandas matplotlib opencv-python scikit-image scikit-learn tqdm torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity
)


import torch
import torch.nn as nn
import torch.optim as optim
import torch_directml

from torch.utils.data import Dataset, DataLoader

device = torch_directml.device()

print("PyTorch version:", torch.__version__)
print("Using device:", device)
print("All imports completed successfully!")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Somesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# Project directory
PROJECT_DIR = Path.cwd()

DATASET_DIR = PROJECT_DIR / "datasets"
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = OUTPUT_DIR / "models"
ENHANCED_DIR = OUTPUT_DIR / "enhanced_images"

# Create output directories
for directory in [OUTPUT_DIR, MODEL_DIR, ENHANCED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# DirectML device for AMD GPU
DEVICE = torch_directml.device()

# Image configuration
IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3

print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATASET_DIR)
print("Output directory:", OUTPUT_DIR)
print("Training device:", DEVICE)


In [ ]:
def load_fingerprint(image_path, image_size=IMAGE_SIZE):
    """
    Load a fingerprint image in grayscale,
    resize it, and normalize pixel values to [0, 1].
    """
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

    if image is None:
        raise ValueError(f"Could not load image: {image_path}")

    image = cv2.resize(
        image,
        (image_size, image_size),
        interpolation=cv2.INTER_AREA
    )

    image = image.astype(np.float32) / 255.0

    return image


def denoise_fingerprint(image):
    """
    Apply median filtering to reduce impulse noise.
    Input and output are normalized grayscale images.
    """
    image_uint8 = np.clip(image * 255, 0, 255).astype(np.uint8)

    denoised = cv2.medianBlur(image_uint8, 3)

    return denoised.astype(np.float32) / 255.0


def enhance_contrast(image, clip_limit=2.0, tile_grid_size=(8, 8)):
    """
    Apply CLAHE to improve local contrast.
    """
    image_uint8 = np.clip(image * 255, 0, 255).astype(np.uint8)

    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=tile_grid_size
    )

    enhanced = clahe.apply(image_uint8)

    return enhanced.astype(np.float32) / 255.0

In [ ]:
def image_to_tensor(image):
    """
    Convert a normalized grayscale image (H, W)
    into a float32 tensor (1, H, W).
    """
    image = np.asarray(image, dtype=np.float32)

    if image.ndim != 2:
        raise ValueError("Expected a 2D grayscale image.")

    if not np.isfinite(image).all():
        raise ValueError("Image contains NaN or infinite values.")

    image = np.clip(image, 0.0, 1.0)

    return torch.from_numpy(image).unsqueeze(0)

In [ ]:
# Supported image formats
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# Find images recursively inside datasets/
image_paths = sorted(
    p for p in DATASET_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

print(f"Total fingerprint images found: {len(image_paths)}")

if len(image_paths) == 0:
    print("No images found. Check DATASET_DIR:", DATASET_DIR)
else:
    print("Dataset path:", DATASET_DIR)
    print("First image:", image_paths[0])

In [ ]:
def degrade_fingerprint(clean_image):
    """
    Generate a synthetic degraded fingerprint from a normalized
    grayscale image in [0, 1].
    """
    image = np.clip(clean_image * 255, 0, 255).astype(np.uint8)

    # Random Gaussian blur
    if random.random() < 0.5:
        kernel_size = random.choice([3, 5])
        image = cv2.GaussianBlur(
            image, (kernel_size, kernel_size), 0
        )

    # Random Gaussian noise
    if random.random() < 0.8:
        noise = np.random.normal(0, random.uniform(3, 12), image.shape)
        image = np.clip(image.astype(np.float32) + noise, 0, 255)

    # Random contrast reduction
    if random.random() < 0.5:
        alpha = random.uniform(0.65, 0.95)
        beta = random.uniform(3, 20)
        image = np.clip(alpha * image + beta, 0, 255)

    return image.astype(np.float32) / 255.0

In [ ]:
class FingerprintDataset(Dataset):
    def __init__(self, paths, image_size=IMAGE_SIZE, split="train"):
        self.paths = list(paths)
        self.image_size = image_size
        self.split = split

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        clean = load_fingerprint(
            self.paths[index], self.image_size
        )
        target = clean.copy()

        if self.split == "train":
            degraded = degrade_fingerprint(clean)
        else:
            # Deterministic degradation for validation and testing
            rng = np.random.default_rng(SEED + index)
            image = np.clip(clean * 255, 0, 255).astype(np.uint8)

            image = cv2.GaussianBlur(image, (3, 3), 0)
            noise = rng.normal(0, 5, image.shape)

            image = image.astype(np.float32)
            image = np.clip(0.8 * image + 10, 0, 255)
            degraded = np.clip(
                image + noise, 0, 255
            ).astype(np.float32) / 255.0

        return image_to_tensor(degraded), image_to_tensor(target)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def get_fingerprint_group(path):
    """
    Group likely impressions/variants of the same finger.
    Uses the parent folder and filename prefix before '_'.
    """
    path = Path(path)
    relative_parent = path.parent.relative_to(DATASET_DIR)
    identity_prefix = path.stem.split("_")[0]

    return str(relative_parent / identity_prefix)


groups = np.array([get_fingerprint_group(p) for p in image_paths])
indices = np.arange(len(image_paths))

# 10% test split
split1 = GroupShuffleSplit(
    n_splits=1, test_size=0.10, random_state=SEED
)
trainval_idx, test_idx = next(
    split1.split(indices, groups=groups)
)

# 10% validation of the full dataset:
# 0.10 / 0.90 = 1/9 of the remaining 90%
split2 = GroupShuffleSplit(
    n_splits=1, test_size=1/9, random_state=SEED
)
train_rel, val_rel = next(
    split2.split(
        trainval_idx,
        groups=groups[trainval_idx]
    )
)

train_idx = trainval_idx[train_rel]
val_idx = trainval_idx[val_rel]

train_paths = [image_paths[i] for i in train_idx]
val_paths = [image_paths[i] for i in val_idx]
test_paths = [image_paths[i] for i in test_idx]

print("Training images:", len(train_paths))
print("Validation images:", len(val_paths))
print("Testing images:", len(test_paths))

# Verify that groups don't overlap
train_groups = set(groups[train_idx])
val_groups = set(groups[val_idx])
test_groups = set(groups[test_idx])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print("Identity-group leakage check passed.")

In [ ]:

train_dataset = FingerprintDataset(train_paths, split="train")
val_dataset = FingerprintDataset(val_paths, split="val")
test_dataset = FingerprintDataset(test_paths, split="test")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))


In [ ]:
inputs, targets = next(iter(train_loader))

print("Input batch shape:", inputs.shape)
print("Target batch shape:", targets.shape)
print("Input range:", inputs.min().item(), "to", inputs.max().item())
print("Target range:", targets.min().item(), "to", targets.max().item())

assert inputs.shape == targets.shape
assert inputs.ndim == 4
assert inputs.shape[1] == 1
assert torch.isfinite(inputs).all()
assert torch.isfinite(targets).all()

print("Batch verification passed.")

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base=32):
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        self.enc1 = DoubleConv(in_channels, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.enc4 = DoubleConv(base * 4, base * 8)

        self.bottleneck = DoubleConv(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2)
        self.dec4 = DoubleConv(base * 16, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = DoubleConv(base * 8, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = DoubleConv(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = DoubleConv(base * 2, base)

        self.output = nn.Conv2d(base, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return torch.sigmoid(self.output(d1))

In [ ]:

# Initialize the U-Net model on the AMD GPU using DirectML
model = UNet(in_channels=1, out_channels=1, base=32).to(DEVICE)

# Test a forward pass
inputs, targets = next(iter(train_loader))
inputs = inputs.to(DEVICE)
targets = targets.to(DEVICE)

model.eval()

with torch.no_grad():
    outputs = model(inputs)

print("Training device:", DEVICE)
print("Input shape:", inputs.shape)
print("Output shape:", outputs.shape)
print("Target shape:", targets.shape)
print("Output range:", outputs.min().item(), "to", outputs.max().item())

assert outputs.shape == targets.shape
assert torch.isfinite(outputs).all()

print("U-Net forward-pass verification passed.")


In [ ]:

model = UNet().to(DEVICE)

parameter_count = sum(p.numel() for p in model.parameters())
print(f"Parameters: {parameter_count:,}")
print("Device:", DEVICE)

sample = torch.randn(
    2, 1, IMAGE_SIZE, IMAGE_SIZE,
    device=DEVICE
)

model.eval()

with torch.no_grad():
    prediction = model(sample)

print("Input shape:", sample.shape)
print("Output shape:", prediction.shape)
print("Output range:", prediction.min().item(), "to", prediction.max().item())

assert prediction.shape == sample.shape
assert torch.isfinite(prediction).all()

print("U-Net architecture test passed!")


In [ ]:

from tqdm.auto import tqdm
import torch.nn as nn
import torch.optim as optim

setup_steps = [
    "Initializing loss function",
    "Initializing optimizer",
    "Setting checkpoint path"
]

with tqdm(
    total=len(setup_steps),
    desc="Model Setup",
    unit="step",
    dynamic_ncols=True
) as progress:

    criterion = nn.MSELoss()
    progress.update(1)
    progress.set_postfix(status="Loss function ready")

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )
    progress.update(1)
    progress.set_postfix(status="Optimizer ready")

    BEST_MODEL_PATH = MODEL_DIR / "unet_fingerprint_best.pth"
    progress.update(1)
    progress.set_postfix(status="Checkpoint path ready")

print("\nLoss function: MSE")
print("Optimizer: Adam")
print("Learning rate:", LEARNING_RATE)
print("Checkpoint:", BEST_MODEL_PATH)
print("Setup completed successfully!")


In [ ]:

from tqdm.auto import tqdm
import time
import torch

PILOT_TRAIN_BATCHES = 100
PILOT_VAL_BATCHES = 20

def run_pilot_epoch(loader, training, max_batches, phase):
    model.train(training)

    total_loss = 0.0
    total_samples = 0
    start_time = time.perf_counter()
    num_batches = min(len(loader), max_batches)

    progress = tqdm(
        total=num_batches,
        desc=phase,
        unit="batch",
        dynamic_ncols=True
    )

    try:
        for batch_idx, (inputs, targets) in enumerate(loader):
            if batch_idx >= max_batches:
                break

            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)

            if training:
                optimizer.zero_grad(set_to_none=True)

                predictions = model(inputs)
                loss = criterion(predictions, targets)

                loss.backward()
                optimizer.step()
            else:
                with torch.no_grad():
                    predictions = model(inputs)
                    loss = criterion(predictions, targets)

            batch_size = inputs.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            avg_loss = total_loss / total_samples
            elapsed = time.perf_counter() - start_time

            progress.update(1)
            progress.set_postfix(
                batch_loss=f"{loss.item():.5f}",
                avg_loss=f"{avg_loss:.5f}",
                elapsed=f"{elapsed:.1f}s"
            )

    finally:
        progress.close()

    return total_loss / max(total_samples, 1)


train_loss = run_pilot_epoch(
    train_loader, True, PILOT_TRAIN_BATCHES, "Training"
)

val_loss = run_pilot_epoch(
    val_loader, False, PILOT_VAL_BATCHES, "Validation"
)

print(f"\nPilot training loss:   {train_loss:.6f}")
print(f"Pilot validation loss: {val_loss:.6f}")
print("Pilot completed successfully!")


In [ ]:

import time
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

NUM_EPOCHS = 30
PATIENCE = 5

BEST_MODEL_PATH = MODEL_DIR / "unet_fingerprint_best.pth"
history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):
    epoch_start = time.perf_counter()

    # -------- TRAINING --------
    model.train()
    train_total_loss = 0.0
    train_samples = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [Train]",
        unit="batch",
        dynamic_ncols=True
    )

    for inputs, targets in train_bar:
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        predictions = model(inputs)
        loss = criterion(predictions, targets)

        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        train_total_loss += loss.item() * batch_size
        train_samples += batch_size

        train_bar.set_postfix(
            loss=f"{loss.item():.5f}",
            avg=f"{train_total_loss / train_samples:.5f}"
        )

    train_loss = train_total_loss / max(train_samples, 1)

    # -------- VALIDATION --------
    model.eval()
    val_total_loss = 0.0
    val_samples = 0

    val_bar = tqdm(
        val_loader,
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [Val]",
        unit="batch",
        dynamic_ncols=True
    )

    with torch.no_grad():
        for inputs, targets in val_bar:
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)

            predictions = model(inputs)
            loss = criterion(predictions, targets)

            batch_size = inputs.size(0)
            val_total_loss += loss.item() * batch_size
            val_samples += batch_size

            val_bar.set_postfix(
                loss=f"{loss.item():.5f}",
                avg=f"{val_total_loss / val_samples:.5f}"
            )

    val_loss = val_total_loss / max(val_samples, 1)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # -------- SAVE BEST MODEL --------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
        }, BEST_MODEL_PATH)

        checkpoint_status = "Best model saved"
    else:
        epochs_without_improvement += 1
        checkpoint_status = "No improvement"

    elapsed = time.perf_counter() - epoch_start

    print(
        f"\nEpoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {elapsed:.1f}s | "
        f"{checkpoint_status}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after epoch {epoch + 1}.")
        break

print("\nTraining finished!")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Best checkpoint: {BEST_MODEL_PATH}")


In [ ]:

plt.figure(figsize=(9, 5))

plt.plot(
    range(1, len(history["train_loss"]) + 1),
    history["train_loss"],
    marker="o",
    label="Training Loss"
)
plt.plot(
    range(1, len(history["val_loss"]) + 1),
    history["val_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("U-Net Fingerprint Enhancement — Training History")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()

print("Best checkpoint loaded!")
print("Best epoch:", checkpoint["epoch"])
print(f"Best validation loss: {checkpoint['val_loss']:.6f}")
print("Model device:", DEVICE)


In [ ]:

# Take one held-out test image
test_dataset = FingerprintDataset(test_paths, split="test")
degraded_tensor, target_tensor = test_dataset[0]

model.eval()

with torch.no_grad():
    enhanced_tensor = model(
        degraded_tensor.unsqueeze(0).to(DEVICE)
    ).detach().cpu().squeeze(0)

degraded = degraded_tensor.squeeze().numpy()
original = target_tensor.squeeze().numpy()
enhanced = enhanced_tensor.squeeze().numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(degraded, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Degraded Input")

axes[1].imshow(enhanced, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("U-Net Output")

axes[2].imshow(original, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Original Target")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()
